<a href="https://colab.research.google.com/github/GunikaNagpal/Bioinformatics_assignments/blob/main/Pairwise_Alignment/hbb_alignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment: Pairwise Sequence Alignment

## Part 1: Optimal Local Alignment — Smith-Waterman
**Sequences:** Row = `C C A T A C G A` | Column = `C A G C T A G C G`

| Parameter | Value |
|---|---|
| Match | +1 |
| Mismatch | −1 |
| Gap | −1 |

The blue path in the matrix shows the optimal local alignment. Max score = **3**. Traceback stops at 0.

In [1]:
# Part 1: Optimal Local Alignment from Smith-Waterman matrix (Slide #5, Day-09)
#
# Sequences:
#   Sequence 1 (row):    C C A T A C G A
#   Sequence 2 (column): C A G C T A G C G
#
# Parameters: match = +1, mismatch = -1, gap = -1
#
# The Smith-Waterman matrix (blue path) has maximum score = 3
# Traceback begins at max score (3) and ends when 0 is encountered.
#
# Tracing the blue highlighted path (score 3 -> 2 -> 1 -> 0):
#   Position of score 3: row G, col G  -> G vs G (match)
#   Position of score 2: row C, col C  -> C vs C (match)
#   Position of score 1: row A, col A  -> A vs A (match)
#
# Optimal Local Alignment:
# ACG
# ACG
#
# All 3 positions are matches => local alignment score = 3

print('Part 1 — Optimal Local Alignment (Smith-Waterman, Slide #5)')
print('Seq1 (row):    C C A T A C G A')
print('Seq2 (column): C A G C T A G C G')
print()
print('Optimal local alignment (traceback from max score = 3):')
print('Seq1: ACG')
print('Seq2: ACG')
print('Alignment score = 3')

Part 1 — Optimal Local Alignment (Smith-Waterman, Slide #5)
Seq1 (row):    C C A T A C G A
Seq2 (column): C A G C T A G C G

Optimal local alignment (traceback from max score = 3):
Seq1: ACG
Seq2: ACG
Alignment score = 3


---
## Part 2: Pairwise Alignment of HBB Genes (Biopython)

In [2]:
!pip install biopython -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 33.9 MB/s eta 0:00:00


In [3]:
from Bio import Entrez, SeqIO
from Bio import Align

Entrez.email = 'your_email@example.com'  # replace with your email

In [4]:
# Step 2a: Fetch HBB gene sequences directly from NCBI via Entrez

accessions = {
    'Human':      'NM_000518.5',
    'Mouse':      'NM_008220.2',
    'Chimpanzee': 'XM_016928727.2'
}

seqs = {}
for species, acc in accessions.items():
    handle = Entrez.efetch(db='nucleotide', id=acc, rettype='fasta', retmode='text')
    record = SeqIO.read(handle, 'fasta')
    handle.close()
    seqs[species] = str(record.seq)
    print(f'{species} ({acc}): {len(record.seq)} bp')

print('\nAll sequences fetched.')

Human (NM_000518.5): 628 bp
Mouse (NM_008220.2): 630 bp
Chimpanzee (XM_016928727.2): 3818 bp

All sequences fetched.


In [5]:
# Configure PairwiseAligner
aligner = Align.PairwiseAligner()
aligner.match_score     =  2
aligner.mismatch_score  = -1
aligner.open_gap_score  = -2
aligner.extend_gap_score = -0.5

print('Aligner ready.')
print(f'match={aligner.match_score}, mismatch={aligner.mismatch_score}, gap open={aligner.open_gap_score}, gap extend={aligner.extend_gap_score}')

Aligner ready.
match=2.0, mismatch=-1.0, gap open=-2.0, gap extend=-0.5


In [6]:
# Step 2b: Global Alignment (Needleman-Wunsch)
# Note: XM_016928727.2 (Chimpanzee) is 3818 bp vs Human 628 bp.
# For a fair global comparison, we align using the full sequences as fetched.
aligner.mode = 'global'

# i. Human vs Mouse
global_score_HM = aligner.score(seqs['Human'], seqs['Mouse'])
alignments_hm_g = aligner.align(seqs['Human'], seqs['Mouse'])
print('=== GLOBAL ALIGNMENT ===')
print(f'Human vs Mouse score:      {global_score_HM:.1f}')
print('Top alignment (Human vs Mouse):')
print(next(iter(alignments_hm_g)))

# ii. Human vs Chimpanzee
global_score_HC = aligner.score(seqs['Human'], seqs['Chimpanzee'])
alignments_hc_g = aligner.align(seqs['Human'], seqs['Chimpanzee'])
print(f'Human vs Chimpanzee score: {global_score_HC:.1f}')
print('Note: Chimpanzee accession XM_016928727.2 is 3818 bp (full mRNA with long UTRs)')
print('      vs Human NM_000518.5 at 628 bp, causing heavy gap penalties in global mode.')
print('Top alignment (Human vs Chimpanzee):')
print(next(iter(alignments_hc_g)))

=== GLOBAL ALIGNMENT ===
Human vs Mouse score:      863.5
Top alignment (Human vs Mouse):
target            0 ----ACATTTGCTTCTGACACAACTGTGTTCACTAGCAACCTC--AAACAGACACCATGG
                  0 ----||.|||||||||||..|...||||||.|||.||||||||--|||||||||.|||||
query             0 GTTTACGTTTGCTTCTGATTCTGTTGTGTTGACTTGCAACCTCAGAAACAGACATCATGG

target           54 TGCATCTGACTCCTGAGGAGAAGTCTGCCGTTACTGCCCTGTGGGGCAAGGTGAACGTGG
                 60 ||||.||||||..||..||||||.||||.||..|||.|||||||||.||||||||||..|
query            60 TGCACCTGACTGATGCTGAGAAGGCTGCTGTCTCTGGCCTGTGGGGAAAGGTGAACGCCG

target          114 ATGAAGTTGGTGGTGAGGCCCTGGGCAGGCTGCTGGTGGTCTACCCTTGGACCCAGAGGT
                120 |||||||||||||||||||||||||||||||||||||.||||||||||||||||||.|||
query           120 ATGAAGTTGGTGGTGAGGCCCTGGGCAGGCTGCTGGTTGTCTACCCTTGGACCCAGCGGT

target          174 TCTTTGAGTCCTTTGGGGATCTGTCCACT-CCTGATGCTGTTATGGGCAACCCTAAGGTG
                180 .||||||...||||||.||.||.|||.||-|||-.||||.|.|||||.||..|.||.|||
query           

In [7]:
# Step 2b: Local Alignment
aligner.mode = 'local'

# i. Human vs Mouse
local_score_HM = aligner.score(seqs['Human'], seqs['Mouse'])
alignments_hm_l = aligner.align(seqs['Human'], seqs['Mouse'])

print('=== LOCAL ALIGNMENT ===')
print(f'Human vs Mouse score: {local_score_HM:.1f}')
print('Top alignment (Human vs Mouse):')
print(next(iter(alignments_hm_l)))

# ii. Human vs Chimpanzee
local_score_HC = aligner.score(seqs['Human'], seqs['Chimpanzee'])
alignments_hc_l = aligner.align(seqs['Human'], seqs['Chimpanzee'])

print(f'Human vs Chimpanzee score: {local_score_HC:.1f}')
print('Top alignment (Human vs Chimpanzee):')
print(next(iter(alignments_hc_l)))

=== LOCAL ALIGNMENT ===
Human vs Mouse score: 869.5
Top alignment (Human vs Mouse):
target            0 ACATTTGCTTCTGACACAACTGTGTTCACTAGCAACCTC--AAACAGACACCATGGTGCA
                  0 ||.|||||||||||..|...||||||.|||.||||||||--|||||||||.|||||||||
query             4 ACGTTTGCTTCTGATTCTGTTGTGTTGACTTGCAACCTCAGAAACAGACATCATGGTGCA

target           58 TCTGACTCCTGAGGAGAAGTCTGCCGTTACTGCCCTGTGGGGCAAGGTGAACGTGGATGA
                 60 .||||||..||..||||||.||||.||..|||.|||||||||.||||||||||..|||||
query            64 CCTGACTGATGCTGAGAAGGCTGCTGTCTCTGGCCTGTGGGGAAAGGTGAACGCCGATGA

target          118 AGTTGGTGGTGAGGCCCTGGGCAGGCTGCTGGTGGTCTACCCTTGGACCCAGAGGTTCTT
                120 |||||||||||||||||||||||||||||||||.||||||||||||||||||.|||.|||
query           124 AGTTGGTGGTGAGGCCCTGGGCAGGCTGCTGGTTGTCTACCCTTGGACCCAGCGGTACTT

target          178 TGAGTCCTTTGGGGATCTGTCCACT-CCTGATGCTGTTATGGGCAACCCTAAGGTGAAGG
                180 |||...||||||.||.||.|||.||-|||-.||||.|.|||||.||..|.||.|||||||
query           184 TG

In [8]:
# Step 2b-iii: Which species is closer to Human?

print('=== ALIGNMENT SCORE SUMMARY ===')
print(f'{"Comparison":<30} {"Global":>10} {"Local":>10}')
print('-' * 52)
print(f'{"Human vs Mouse":<30} {global_score_HM:>10.1f} {local_score_HM:>10.1f}')
print(f'{"Human vs Chimpanzee":<30} {global_score_HC:>10.1f} {local_score_HC:>10.1f}')

closer_l = 'Chimpanzee' if local_score_HC > local_score_HM else 'Mouse'
print(f'\nLocal alignment => closer to Human: {closer_l}')

# Answer to 2.iii:
# Based on LOCAL alignment scores:
#   Human vs Mouse local score:      869.5
#   Human vs Chimpanzee local score: 446.5
# The HBB gene closest to Human based on computed alignment scores is Mouse (NM_008220.2),
# as it yields a higher local alignment score (869.5) than Chimpanzee (446.5).
# Note: The Chimpanzee accession XM_016928727.2 is a 3818 bp full mRNA (with long UTRs),
# whereas the Human sequence NM_000518.5 is only 628 bp. This length mismatch artificially
# lowers the Chimpanzee score. Biologically, Chimpanzee is the closer relative to Human
# (~98-99% DNA identity), but based purely on these alignment scores, Mouse scores higher.


=== ALIGNMENT SCORE SUMMARY ===
Comparison                         Global      Local
----------------------------------------------------
Human vs Mouse                      863.5      869.5
Human vs Chimpanzee                -670.5      446.5

Local alignment => closer to Human: Mouse
